# Día 4 - Clustering y segmentación económica

**Fecha:** 16-10-2026

Hoy no tenemos una respuesta correcta `y`. Queremos descubrir estructura en los datos.

## Objetivos

- entender distancia, escala y centroides;
- aplicar K-Means;
- elegir K con elbow y silhouette;
- aplicar clustering jerárquico;
- interpretar un dendrograma;
- construir perfiles económicos de clusters;
- evitar confundir “grupo parecido” con causalidad o categoría oficial.

## 0. Librerías y carpeta de datos

No hay instalación automática. Si una librería opcional falta, el notebook mostrará el comando que puedes ejecutar manualmente en una celda nueva. Todos los imports principales quedan juntos para que sea claro qué herramientas usa la jornada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

In [ ]:
DATA_DIR = Path('../datos')
if not DATA_DIR.exists():
    DATA_DIR = Path('datos')
print('Datos:', DATA_DIR.resolve())

A partir de aquí, cada bloque de código irá acompañado de una explicación de **qué entra, qué sale y qué debemos mirar**. Ejecutar una celda sin interpretar el resultado no cuenta como comprender el modelo.

# 1. Cargar indicadores para países de América Latina

Intentamos construir una tabla desde datos públicos del Banco Mundial. Si falla Internet usamos `latam_indicadores_fallback.csv`, marcado como sintético.

La unidad de observación será **un país** y las columnas serán indicadores comparables.

## ¿Qué representa esta tabla?

Aquí cada fila representa **un país** y cada columna un indicador socioeconómico. La pregunta de clustering será descriptiva:

> ¿qué países resultan parecidos según el conjunto de indicadores elegido?

El resultado depende de **qué variables incluimos, cómo las escalamos y qué distancia usa el algoritmo**. Por eso un cluster no es una “verdad natural” descubierta automáticamente.

In [ ]:

paises = {
    'ECU':'Ecuador','COL':'Colombia','PER':'Perú','CHL':'Chile','ARG':'Argentina',
    'BRA':'Brasil','URY':'Uruguay','PRY':'Paraguay','BOL':'Bolivia','MEX':'México'
}

indicadores = {
    'NY.GDP.PCAP.CD':'pib_per_capita_usd',
    'FP.CPI.TOTL.ZG':'inflacion_pct',
    'SL.UEM.TOTL.ZS':'desempleo_pct',
    'NE.EXP.GNFS.ZS':'exportaciones_pct_pib',
    'NE.IMP.GNFS.ZS':'importaciones_pct_pib',
    'IT.NET.USER.ZS':'internet_pct_poblacion'
}

def indicador_varios_paises(indicador, start=2018, end=2025):
    # La API admite varios países separados por ';'.
    # Así hacemos UNA petición por indicador en lugar de una por país.
    codigos=';'.join(paises.keys())
    url=f'https://api.worldbank.org/v2/country/{codigos}/indicator/{indicador}'
    r=requests.get(url,params={'format':'json','date':f'{start}:{end}','per_page':1000},timeout=4)
    r.raise_for_status()
    filas=r.json()[1]

    # Nos quedamos con el valor más reciente no nulo de cada país.
    salida={}
    for fila in filas:
        cod=fila.get('countryiso3code')
        val=fila.get('value')
        if cod in paises and val is not None and cod not in salida:
            salida[cod]=val
    return salida

In [ ]:
try:
    # Partimos de una fila por país.
    latam=pd.DataFrame({'codigo':list(paises.keys())})
    latam['pais']=latam['codigo'].map(paises)

    # Solo 6 peticiones HTTP: una por indicador.
    for ind,nombre_col in indicadores.items():
        valores=indicador_varios_paises(ind)
        latam[nombre_col]=latam['codigo'].map(valores)

    latam=latam.drop(columns='codigo').dropna()
    fuente='Banco Mundial API'
except Exception as e:
    print('API no disponible:', e)
    latam=pd.read_csv(DATA_DIR/'latam_indicadores_fallback.csv')
    fuente='Respaldo SINTÉTICO DIDÁCTICO'

print('Fuente:',fuente)
latam

# 2. Entender antes de clusterizar

Preguntas obligatorias:

- ¿cada columna usa la misma escala?
- ¿hay variables altamente correlacionadas?
- ¿hay outliers?
- ¿qué significa “parecido” en este contexto?

K-Means usa distancia euclidiana. Sin escalado, PIB per cápita podría dominar completamente a inflación por tener magnitudes mayores.

Antes de K-Means revisa tres niveles:

1. **semántica:** ¿los indicadores miden cosas comparables y del mismo periodo?
2. **calidad:** ¿hay faltantes/outliers?
3. **geometría:** ¿alguna columna domina por escala o está casi duplicada por otra?

En no supervisado estas decisiones son aún más importantes porque no existe una `y` que nos diga si “acertamos”.

In [ ]:
features = [c for c in latam.columns if c not in ['pais','tipo_dato']]

latam[features].describe().T

### Cómo leer esta tabla

Compara `std`, `min` y `max` entre indicadores. PIB per cápita puede estar en miles de dólares, inflación en porcentajes y esperanza de vida en años. Si calculáramos distancias directamente, las unidades grandes dominarían.

In [ ]:
# Correlación para detectar variables que contienen información parecida.
latam[features].corr().round(2)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(latam[features])

# Después del escalado, cada columna queda aproximadamente en la misma escala.
pd.DataFrame(X_scaled, columns=features).describe().round(2)

## 2.1 Ver la escala antes y después

Una forma de demostrar por qué escalamos es comparar rangos. Antes de `StandardScaler`, PIB per cápita y tasas porcentuales viven en órdenes de magnitud distintos.

`StandardScaler` transforma cada columna a media aproximadamente 0 y desviación 1:

$$z=\frac{x-\mu}{\sigma}$$

Eso **no hace que todas las variables sean igual de importantes conceptualmente**. Solo evita que la unidad de medida determine la distancia.

In [ ]:
rangos=pd.DataFrame({
    'min':latam[features].min(),
    'max':latam[features].max(),
    'rango':latam[features].max()-latam[features].min()
})
rangos.round(2)

## 2.2 ¿Qué distancia está minimizando K-Means?

La asignación usa distancia euclidiana. Por eso:

- escalas importan;
- outliers pueden arrastrar centroides;
- categorías sin codificación numérica apropiada no encajan directamente;
- clusters aproximadamente esféricos son un escenario favorable.

K-Means asigna cada observación al centroide más cercano y busca minimizar la suma de distancias cuadradas dentro de clusters:

$$Inercia=\sum_{k=1}^{K}\sum_{x_i\in C_k}\|x_i-\mu_k\|^2$$

Por eso:

- variables escaladas importan;
- outliers pueden mover centroides;
- clusters esféricos/compactos encajan mejor con su geometría que formas muy irregulares.

# 3. K-Means paso a paso

Algoritmo conceptual:

1. elegir K centroides iniciales;
2. asignar cada punto al centroide más cercano;
3. recalcular cada centroide como media de su grupo;
4. repetir hasta estabilizar.

La función objetivo que reduce K-Means está relacionada con la suma de distancias cuadradas dentro de cada cluster (*inertia*).

## Una iteración mental de K-Means

1. se eligen centroides iniciales;
2. cada país se asigna al centroide más cercano;
3. cada centroide se recalcula como la media de su grupo;
4. se repite hasta que las asignaciones/centroides dejan de cambiar significativamente.

`random_state` hace reproducible la inicialización, pero el número 42 no tiene ninguna propiedad estadística especial.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_scaled)

resultado = latam.copy()
resultado['cluster'] = clusters
resultado[['pais','cluster']].sort_values('cluster')

In [ ]:
# Perfil medio de cada cluster en las unidades ORIGINALES.
perfil = resultado.groupby('cluster')[features].mean().round(2)
perfil

## 3.1 Centroides en unidades originales

Los centroides internos están en z-scores porque entrenamos con datos estandarizados. Podemos aplicar `inverse_transform` para volver a unidades comprensibles.

Los centroides en escala estandarizada son difíciles de comunicar. Al aplicar `inverse_transform`, podemos volver a dólares, porcentajes o años y describir cada grupo con lenguaje del dominio.

In [ ]:
centroides_originales=pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=features
)
centroides_originales.round(2)

# 4. ¿Cómo elegir K?

No existe una respuesta puramente automática.

Usaremos dos señales:

### Elbow
Mira cómo disminuye `inertia_` al aumentar K. Buscamos un punto donde agregar clusters produce poca mejora adicional.

### Silhouette
Compara cohesión dentro del cluster y separación respecto a otros clusters. Valores más altos suelen indicar mejor separación.

Con pocos países, estas métricas deben interpretarse con cautela.

Para cada observación:

$$s=\frac{b-a}{\max(a,b)}$$

- $a$: distancia media a su propio cluster;
- $b$: distancia media al cluster alternativo más cercano.

$s$ cerca de 1 → bien separado; cerca de 0 → frontera; negativo → quizá está más cerca de otro cluster.

No existe un detector infalible de K.

- **Elbow:** busca el punto donde añadir más clusters deja de reducir mucho la inercia.
- **Silhouette:** compara qué tan cerca está cada observación de su propio cluster frente al cluster vecino.
- **Interpretabilidad:** el grupo debe tener significado útil y no ser un artefacto de una sola variable.
- **Estabilidad:** pequeñas perturbaciones no deberían destruir completamente la solución.

In [ ]:
ks = range(2, min(8, len(latam)))
filas=[]

for k in ks:
    km=KMeans(n_clusters=k,random_state=42,n_init=20)
    lab=km.fit_predict(X_scaled)
    filas.append({
        'k':k,
        'inertia':km.inertia_,
        'silhouette':silhouette_score(X_scaled,lab)
    })

seleccion_k=pd.DataFrame(filas)
seleccion_k

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(seleccion_k['k'], seleccion_k['inertia'], marker='o')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Método del codo')
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(seleccion_k['k'], seleccion_k['silhouette'], marker='o')
plt.xlabel('K')
plt.ylabel('Silhouette')
plt.title('Silhouette por número de clusters')
plt.show()

## 4.1 ¿El mejor Silhouette es automáticamente el K correcto?

No. Un K puede separar geométricamente bien pero producir segmentos poco útiles para la pregunta económica. La elección final combina:

- separación estadística;
- estabilidad;
- tamaño de grupos;
- interpretabilidad;
- finalidad del análisis.

# 5. Visualizar clusters en 2 variables

Un cluster se calcula con todas las features seleccionadas. Un scatter de dos variables es solo una **proyección parcial**.

Una visualización con solo dos variables **no contiene toda la información** usada por K-Means. Sirve para inspeccionar una proyección parcial. No concluyas que dos países son “idénticos” porque sus puntos se vean cercanos en ese gráfico.

In [ ]:
plt.figure(figsize=(8,5))

for c in sorted(resultado['cluster'].unique()):
    subset=resultado[resultado['cluster']==c]
    plt.scatter(
        subset['pib_per_capita_usd'],
        subset['desempleo_pct'],
        s=80,
        label=f'Cluster {c}'
    )
    for _,fila in subset.iterrows():
        plt.annotate(fila['pais'],(fila['pib_per_capita_usd'],fila['desempleo_pct']),fontsize=8)

plt.xlabel('PIB per cápita')
plt.ylabel('Desempleo (%)')
plt.title('Clusters vistos en dos variables')
plt.legend()
plt.show()

## 5.1 Matriz de perfiles tipo heatmap con Matplotlib

Visualizamos z-scores medios por cluster. Positivo significa “por encima de la media global” y negativo “por debajo”.

In [ ]:
z_plot=pd.DataFrame(X_scaled,columns=features)
z_plot['cluster']=resultado['cluster'].to_numpy()
perfil_std=z_plot.groupby('cluster').mean()

fig,ax=plt.subplots(figsize=(10,4))
im=ax.imshow(perfil_std.values,aspect='auto')
ax.set_xticks(range(len(features)),labels=features,rotation=45,ha='right')
ax.set_yticks(range(len(perfil_std)),labels=[f'Cluster {i}' for i in perfil_std.index])
fig.colorbar(im,ax=ax,label='z-score medio')
plt.title('Perfil relativo de cada cluster')
plt.tight_layout()
plt.show()

# 6. Clustering jerárquico

El clustering jerárquico aglomerativo comienza con cada observación separada y va fusionando grupos.

El **dendrograma** muestra a qué distancia se producen las fusiones. Cortar el árbol a distintas alturas produce distintos números de clusters.

El clustering jerárquico aglomerativo empieza con cada observación separada y va fusionando grupos. El **dendrograma** conserva el historial de fusiones.

El método `linkage` define cómo medir distancia entre grupos:

- `single`: vecinos más cercanos;
- `complete`: puntos más lejanos;
- `average`: distancia promedio;
- `ward`: busca minimizar aumento de varianza interna y suele combinarse con Euclidiana.

In [ ]:
# linkage construye la jerarquía. Ward intenta minimizar incremento de varianza interna.
Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(12,5))
dendrogram(Z, labels=latam['pais'].tolist(), leaf_rotation=45)
plt.ylabel('Distancia de fusión')
plt.title('Dendrograma - clustering jerárquico')
plt.show()

In [ ]:
hier = AgglomerativeClustering(n_clusters=3, linkage='ward')
resultado['cluster_jerarquico'] = hier.fit_predict(X_scaled)

resultado[['pais','cluster','cluster_jerarquico']].sort_values('cluster_jerarquico')

## 6.1 Métodos de linkage

`ward`, `complete`, `average` y `single` definen de forma distinta la distancia entre grupos. La estructura del dendrograma puede cambiar.

In [ ]:
# Comparamos silhouette de algunas variantes aglomerativas con 3 clusters.
for linkage_name in ['ward','complete','average','single']:
    m=AgglomerativeClustering(n_clusters=3,linkage=linkage_name)
    lab=m.fit_predict(X_scaled)
    print(linkage_name,'-> silhouette:',round(silhouette_score(X_scaled,lab),3))

## 6.2 Estabilidad: quitar una variable

Si quitar una sola variable cambia casi todos los grupos, la segmentación puede ser frágil. Esta prueba sencilla ayuda a discutir robustez.

Si quitar una sola variable cambia totalmente los clusters, la segmentación es frágil. Esa sensibilidad es información importante: quizá una feature domina o el dataset es demasiado pequeño para sostener una estructura estable.

In [ ]:
features_sin_pib=[c for c in features if c!='pib_per_capita_usd']
X2=StandardScaler().fit_transform(latam[features_sin_pib])
lab2=KMeans(n_clusters=3,n_init=20,random_state=42).fit_predict(X2)

pd.DataFrame({
    'pais':latam['pais'],
    'cluster_todas':resultado['cluster'],
    'cluster_sin_pib':lab2
})

# 7. Interpretar clusters, no solo numerarlos

“Cluster 0” no tiene significado económico por sí mismo. Debemos construir un perfil.

Ejemplo de redacción:

> Cluster 1 agrupa países con mayor PIB per cápita y penetración de Internet, pero no necesariamente menor inflación.

No debemos escribir:

> “pertenecer al cluster 1 causa mayor desarrollo”.

Clustering describe semejanza bajo las variables y transformaciones elegidas.

Un número de cluster (`0`, `1`, `2`) no tiene significado por sí mismo. Hay que construir perfiles:

- promedios/medianas por indicador;
- países que pertenecen al grupo;
- variables que más diferencian grupos;
- excepciones dentro de cada cluster.

Y recuerda: **segmentar no implica causalidad**. Un cluster “alta inflación / bajo crecimiento” no demuestra que una variable cause la otra.

In [ ]:
# Z-score medio por cluster: muestra qué variables están por encima/debajo de la media global.
z = pd.DataFrame(X_scaled, columns=features)
z['cluster']=resultado['cluster'].to_numpy()

perfil_z = z.groupby('cluster').mean().round(2)
perfil_z

## 7.1 Fuentes regionales: BCE, INEC y CEPALSTAT

El mismo flujo sirve para datos descargados manualmente o APIs institucionales:

```text
fuente -> DataFrame -> tipos/nulos -> selección de indicadores -> escala -> clustering -> perfil
```

Cuando una fuente entrega CSV/Excel, no hace falta inventar una integración compleja: primero se descarga, se documenta la fecha/corte y se trabaja localmente. Si existe API, puede automatizarse después.

In [ ]:
# Plantilla genérica para un CSV descargado de una fuente pública.
# Cambia la ruta y los nombres de columnas según el archivo real.
# datos_fuente = pd.read_csv(DATA_DIR / 'archivo_descargado.csv')
# datos_fuente.info()
# datos_fuente.head()

print('Plantilla lista: sustituir archivo y columnas según la fuente elegida.')

# Taller U3

1. Elige entre K=2,3,4 usando elbow + silhouette + interpretación.
2. Describe cada cluster en lenguaje económico.
3. Indica qué variables parecen separar más los grupos.
4. Quita una variable y vuelve a clusterizar. ¿Cambian los grupos?
5. Explica una limitación: escala, año de los indicadores, tamaño de muestra, correlación, etc.

La entrega debe contener **interpretación**, no solo código y números de cluster.

## Preguntas para cerrar

1. ¿Por qué escalar antes de K-Means?
2. ¿Qué representa un centroide?
3. ¿Por qué inercia siempre baja al aumentar K?
4. ¿Qué información añade Silhouette?
5. ¿Por qué un scatter de dos variables no muestra todas las dimensiones usadas?
6. ¿Qué diferencia conceptual hay entre K-Means y clustering jerárquico?
7. ¿Cómo comprobarías que una segmentación económica es estable y útil?